In [1]:
import pandas as pd
import re

In [2]:
path = f'C:/Users/Angelo/Documents/github/capstone_project/data/raw/titanic.csv'

In [3]:
# load data
def load_data(path):
    df = pd.read_csv(path)
    return df

In [4]:
# copy loaded data
def copy_data(df):
    df_copy = df.copy()
    return df_copy

In [5]:
# execute
df = copy_data(load_data(path))

In [6]:
# view data
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [7]:
# define raw column categories
categorical_raw = df.select_dtypes(include=['object', 'category','str']).columns.tolist()
numerical_raw = df.select_dtypes(include=['float64', 'int64', 'float32', 'int32']).columns.tolist()

In [8]:
# check value range
# numerical cols
#df[numerical_raw[0]].describe(include='all') # PassengerId ✅
#df[numerical_raw[1]].describe(include='all') # Survived ✅
#df[numerical_raw[2]].describe(include='all') # Pclass ✅
# df[numerical_raw[3]].describe(include='all') # Age ✅
# df[numerical_raw[4]].describe(include='all') # SibSp ✅
# df[numerical_raw[5]].describe(include='all') # Parch ✅
# df[numerical_raw[6]].describe(include='all') # Fare ✅

# categorical cols
# df[categorical_raw[0]].describe(include='all') # Name ✅
# df[categorical_raw[1]].describe(include='all') # Sex ✅
# df[categorical_raw[2]].describe(include='all') # Ticket ✅ - drop
# df[categorical_raw[3]].describe(include='all') # Cabin ✅ - split
# df[categorical_raw[4]].describe(include='all') # Embarked ✅


IndexError: list index out of range

In [40]:
# drop Ticket
df = df.drop('Ticket')

KeyError: "['Ticket'] not found in axis"

In [93]:
# split col
def return_n_elements(element, n_returns):
    return pd.Series([element] * n_returns)

def get_first_value_in_col(
    df: pd.DataFrame,
    col: str,
    split_by: str
    ):
    '''gets the first value in an element, returns df'''
    def get_first_in_element(element: str):
        # case 0: check if na
        if pd.isna(element): 
            return return_n_elements(element, 1)

        # case 1: one or more values
        # get first value from list
        first_value = str(element).split(split_by)[0]

        return pd.Series([first_value])

    # apply element-wise function to all of column
    df[[f'{col}_first_value']] = df[col].apply(get_first_in_element)
    return df[[f'{col}_first_value']]

def split_alphanumeric(
    df: pd.DataFrame,
    col: str):

    def splitter(element: str):
        '''splits element into letter(s) and number(s) elements'''
        # case 0: check if na
        if pd.isna(element):
            return return_n_elements(element, 2)

        # verify cabine matches pattern:(letter(s))(number(s)) or (letter(s))(number(s)):
        # 4 groups are outputted
        match = re.match(r'^([A-Za-z]+)(\d+)$|^(\d+)([A-Za-z]+)$', element)
        
        # return NA if NA
        if not match:
            return return_n_elements(pd.NA, 2)
        
        # return as two series
        # save groups as dtype var options 
        str1, num1, num2, str2 = match.groups()
        
        # if letter(s) number(s)
        if str1 is not None:
            return pd.Series([str1, int(num1)])
 
        # else number(s) letter(s)
        return pd.Series([int(num2), str2])

    # apply element-wise function to all of cabin column
    df[[f'{col}_left', f'{col}_right']] = df[col].apply(splitter)

    return df

    alphanumeric alphanumeric_first_value alphanumeric_first_value_left  \
0  ABC123 DEF456                   ABC123                           ABC   
1            NaN                      NaN                           NaN   
2         123ABC                   123ABC                           123   
3  123ABC DEF321                   123ABC                           123   

  alphanumeric_first_value_right  
0                            123  
1                            NaN  
2                            ABC  
3                            ABC  


In [ ]:
#drop missing
clean_df = df.dropna()

In [54]:
# verify done
clean_df.info()

<class 'pandas.DataFrame'>
Index: 84 entries, 12 to 414
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   PassengerId    84 non-null     int64  
 1   Survived       84 non-null     int64  
 2   Pclass         84 non-null     int64  
 3   Name           84 non-null     str    
 4   Sex            84 non-null     str    
 5   Age            84 non-null     float64
 6   SibSp          84 non-null     int64  
 7   Parch          84 non-null     int64  
 8   Ticket         84 non-null     str    
 9   Fare           84 non-null     float64
 10  Cabin          84 non-null     str    
 11  Embarked       84 non-null     str    
 12  cabin_letters  84 non-null     str    
 13  cabin_numbers  84 non-null     float64
dtypes: float64(3), int64(5), str(6)
memory usage: 14.1 KB


In [ ]:
#pipeline
def pipeline(path):
    df = copy_data(load_data(path)) # load and copy df
    df = df.drop('Ticket')
    # split Cabin
    df[['cabin_letters', 'cabin_numbers']] = df['Cabin'].apply(split_cabin_into_letters_col_and_nums_col)
    df['cabin_letters'] = df['cabin_letters'].astype(str) 
    df['cabin_numbers'] = pd.to_numeric(df['cabin_numbers'], errors='coerce')
    clean_df = df.dropna()
    return clean_df